# Sentinel-GenAI: Enterprise Hybrid RAG and Databricks Genie Semantic Engine

### The Enterprise GenAI Challenge in Modern Lakehouses:
Enterprise stakeholders need answers that span both numerical analytics and qualitative governance:
- Traditional document RAG fails because it cannot aggregate across millions of rows in Delta Lake.
- Traditional Text-to-SQL fails because it cannot interpret SLAs, contractual penalty clauses, or compliance guidelines.

### The Sentinel-GenAI Solution:
- **Structure-Aware Document Chunker**: Preserves Markdown tables and code blocks without slicing boundaries.
- **Hybrid Mosaic AI Vector Search**: Dense vector similarity combined with sparse BM25 scoring via Reciprocal Rank Fusion (RRF).
- **Databricks Genie Semantic Compiler**: Generates table descriptions, column constraints, join graphs, and a verified Trusted SQL Catalog.
- **Compound Query Router & Evaluator**: Intelligently classifies query intent, dispatches to SQL or Vector indices, and produces grounded, citation-backed answers with automated evaluation scorecards.

In [0]:
# COMMAND ----------
# MAGIC %pip install .
# COMMAND ----------
from sentinel.genai import (
    StructureAwareChunker,
    LocalDeterministicEmbeddings,
    HybridVectorIndex,
    GenieSemanticCompiler,
    TableAnnotation,
    ColumnAnnotation,
    TableRelationship,
    TrustedQuery,
    CompoundQueryRouter,
    GenAIEvaluator,
    EvalTestCase,
    QueryIntent,
)

print('Sentinel-GenAI modules imported successfully.')

## 1. Structure-Aware Chunking & Hybrid Vector Search Indexing
We chunk enterprise governance assets (SLA contracts, data dictionaries) while strictly preserving Markdown tables.

In [0]:
sla_document = '''# Enterprise Service Level Agreement (SLA)

| Service Tier | Uptime Target | Reconciliation Grace Period | Down-Time Penalty Rate |
| :--- | :--- | :--- | :--- |
| Platinum | 99.99% | 15 Minutes | $10,000 per hour |
| Gold | 99.90% | 1 Hour | $2,500 per hour |
| Silver | 99.00% | 4 Hours | $500 per hour |

### Breach Escalation Protocol
When pipeline degradation breaches SLA thresholds, the incident response dispatcher notifies platform leads via PagerDuty and quarantines affected partitions.'''

chunker = StructureAwareChunker(max_chunk_chars=300, overlap_chars=30, preserve_tables=True)
chunks = chunker.chunk_document('doc-sla-2026', sla_document, doc_metadata={'tier': 'enterprise'})

embedder = LocalDeterministicEmbeddings(dimension=64)
vector_index = HybridVectorIndex(embedding_provider=embedder, rrf_k=60)
vector_index.index_chunks(chunks)

print(f'Indexed {len(chunks)} chunks into Hybrid Vector Search.')

## 2. Compile Databricks Genie Semantic Layer & Trusted SQL Catalog
We annotate Gold dimensional models and register verified, gold-standard SQL queries into the semantic space.

In [0]:
compiler = GenieSemanticCompiler(
    space_name='Sentinel Orders & Governance Space',
    space_description='Governed Lakehouse semantic space for analytical queries and SLA compliance.'
)

fact_orders = TableAnnotation(
    catalog='sentinel_prod',
    schema='gold',
    table_name='fact_orders',
    description='Gold fact table containing validated, deduplicated customer orders.',
    primary_keys=['order_id'],
    cluster_by=['order_date'],
    columns=[
        ColumnAnnotation(name='order_id', data_type='string', description='Primary key', semantic_role='primary_key'),
        ColumnAnnotation(name='order_date', data_type='date', description='Order date', semantic_role='timestamp'),
        ColumnAnnotation(name='total_amount', data_type='decimal(18,2)', description='Net revenue', semantic_role='metric'),
    ]
)
compiler.register_table(fact_orders)

compiler.register_trusted_query(
    TrustedQuery(
        query_id='monthly_gross_revenue',
        metric_name='Gross Revenue',
        description='Aggregates total net order revenue by month.',
        example_prompts=['Show monthly revenue', 'What is our monthly sales volume?'],
        sql_template='SELECT date_trunc(\'month\', order_date) AS order_month, sum(total_amount) AS monthly_revenue FROM sentinel_prod.gold.fact_orders GROUP BY 1 ORDER BY 1 DESC',
        target_tables=['sentinel_prod.gold.fact_orders']
    )
)

# Generate Unity Catalog comment statements
ddl = compiler.generate_unity_catalog_comments()
print(f'Generated {len(ddl)} Unity Catalog semantic DDL statements.')

## 3. End-to-End Compound Query Execution: Hybrid Analytical & Policy Synthesis
Run a hybrid prompt requiring both analytical revenue aggregation and SLA penalty terms.

In [0]:
router = CompoundQueryRouter(vector_index=vector_index, semantic_compiler=compiler)

hybrid_prompt = 'What is our monthly gross revenue, and what are the Platinum SLA downtime penalties?'
response = router.route_and_execute(hybrid_prompt)

print('=== PROMPT ===')
print(response.prompt)
print('\n=== CLASSIFIED INTENT ===')
print(response.intent.value)
print('\n=== SYNTHESIZED HYBRID RESPONSE ===')
print(response.answer)

## 4. Run Automated Evaluation Harness & Generate Scorecard
Benchmark intent classification accuracy, context relevance, and guardrail verification across test cases.

In [0]:
evaluator = GenAIEvaluator(router=router)

suite = [
    EvalTestCase(
        prompt='Show monthly revenue summary',
        expected_intent=QueryIntent.ANALYTICAL,
        expected_keywords=['monthly_revenue']
    ),
    EvalTestCase(
        prompt='What are the Platinum SLA penalty rates for pipeline downtime?',
        expected_intent=QueryIntent.POLICY_RETRIEVAL,
        expected_keywords=['10,000', 'uptime']
    ),
    EvalTestCase(
        prompt='Calculate monthly sales and check escalation protocols under SLA',
        expected_intent=QueryIntent.HYBRID,
        expected_keywords=['monthly_revenue', 'escalation']
    ),
]

scorecard = evaluator.evaluate(suite)
print(scorecard.to_markdown())